In [1]:
from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    count,
    when,
    sum,
    avg,
    min,
    max
)

from pyspark.sql.types import TimestampType

In [2]:
spark = SparkSession.builder \
    .appName("SalesDataCleaning8000") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark Version:", spark.version)
print("Spark Session Created Successfully")


Spark Version: 4.1.2
Spark Session Created Successfully


In [3]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("../Data/celebal_week5_spark_dataset_8000.csv")

print("Dataset Loaded Successfully")


Dataset Loaded Successfully


In [4]:
print("First Five Records:")

df.show(5, truncate=False)


First Five Records:
+-------+----------------+-------+----------------+-----------+---------+---------+---+------------+-------------------+--------------------+---------+-------+--------+--------+--------+
|user_id|transaction_date|region |product_category|sale_amount|status   |city     |age|subscription|raw_timestamp      |email               |username |price  |store_id|quantity|discount|
+-------+----------------+-------+----------------+-----------+---------+---------+---+------------+-------------------+--------------------+---------+-------+--------+--------+--------+
|USR0915|2025-04-25      |East   |Electronics     |825.55     |Cancelled|Jaipur   |23 |Standard    |2025-04-25 13:02:00|user245@example.com |user_768 |173.8  |STORE07 |5       |0.05    |
|USR0815|2025-08-27      |Central|Groceries       |2041.21    |Completed|Ahmedabad|55 |Basic       |2025-08-27 00:48:00|user1308@example.com|user_3463|2551.51|STORE11 |1       |0.2     |
|USR0380|2025-10-12      |South  |Electronics

In [5]:
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

Total Rows: 8000
Total Columns: 16


In [6]:
print("Column Names:")

print(df.columns)


Column Names:
['user_id', 'transaction_date', 'region', 'product_category', 'sale_amount', 'status', 'city', 'age', 'subscription', 'raw_timestamp', 'email', 'username', 'price', 'store_id', 'quantity', 'discount']


In [7]:
print("Dataset Schema:")

df.printSchema()

Dataset Schema:
root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)



## Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

Traditional MapReduce repeatedly reads intermediate data from disk and writes results back to disk. These frequent disk operations increase processing time, especially for iterative workloads.

Apache Spark improves these limitations by supporting in-memory computation. It can keep intermediate results in memory instead of repeatedly storing them on disk.

Spark also provides easy-to-use APIs such as DataFrames and supports SQL, machine learning, streaming, and graph processing.


## Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

Spark uses in-memory computing to store intermediate data in RAM whenever possible.

Machine learning algorithms often process the same dataset multiple times. Traditional disk-based systems repeatedly read and write data during each iteration, which increases processing time.


## Q3: Remove all duplicate rows based on user_id and transaction_date

The dropDuplicates() function is used to remove duplicate records based on selected columns.

In this operation, duplicate records are identified using the combination of user_id and transaction_date.

In [8]:
rows_before_duplicates = df.count()

df_unique = df.dropDuplicates([
    "user_id",
    "transaction_date"
])

rows_after_duplicates = df_unique.count()

print("Rows Before Duplicate Removal:", rows_before_duplicates)
print("Rows After Duplicate Removal:", rows_after_duplicates)

print(
    "Duplicate Records Removed:",
    rows_before_duplicates - rows_after_duplicates
)


Rows Before Duplicate Removal: 8000
Rows After Duplicate Removal: 7824
Duplicate Records Removed: 176


## Q4: Filter rows where region is West and find the average sale_amount for each product_category

First, records belonging to the West region are filtered.

The filtered records are then grouped according to product category, and the average sale amount is calculated for each category.

In [9]:
west_category_sales = df \
    .filter(col("region") == "West") \
    .groupBy("product_category") \
    .agg(
        avg("sale_amount").alias("average_sale_amount")
    ) \
    .orderBy(
        col("average_sale_amount").desc()
    )

west_category_sales.show()


+----------------+-------------------+
|product_category|average_sale_amount|
+----------------+-------------------+
|       Furniture| 13155.532152777778|
|       Groceries| 13023.983837209305|
|        Clothing| 12836.765969230762|
|           Books| 12723.322537764356|
|     Electronics|  12224.54432055749|
+----------------+-------------------+



## Q5: What is the difference between .na.drop() and .na.fill()?

The .na.drop() operation removes rows containing null values.

The .na.fill() operation replaces null values with a specified value.

For example:

df.na.fill({"status": "Unknown"})

replaces null values in the status column with Unknown.

In [10]:
null_counts = df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
])

null_counts.show(truncate=False)


+-------+----------------+------+----------------+-----------+------+----+---+------------+-------------+-----+--------+-----+--------+--------+--------+
|user_id|transaction_date|region|product_category|sale_amount|status|city|age|subscription|raw_timestamp|email|username|price|store_id|quantity|discount|
+-------+----------------+------+----------------+-----------+------+----+---+------------+-------------+-----+--------+-----+--------+--------+--------+
|0      |0               |0     |0               |0          |120   |0   |0  |0           |0            |80   |70      |90   |0       |0       |0       |
+-------+----------------+------+----------------+-----------+------+----+---+------------+-------------+-----+--------+-----+--------+--------+--------+



In [11]:
df_status_filled = df.na.fill({
    "status": "Unknown"
})

print("Null Status Values After Filling:")

df_status_filled.select(
    count(
        when(col("status").isNull(), "status")
    ).alias("NullStatusCount")
).show()

Null Status Values After Filling:
+---------------+
|NullStatusCount|
+---------------+
|              0|
+---------------+



## Q6: Find the total count of records for each city, but display only cities where the count is greater than 100

The dataset is grouped according to city.

The count() function calculates the number of records for each city, and the filter() operation displays only cities having more than 100 records.

In [12]:
city_counts = df \
    .groupBy("city") \
    .agg(
        count("*").alias("total_records")
    ) \
    .filter(
        col("total_records") > 100
    ) \
    .orderBy(
        col("total_records").desc()
    )

city_counts.show()

+---------+-------------+
|     city|total_records|
+---------+-------------+
|    Delhi|          826|
|   Mumbai|          816|
|Ahmedabad|          810|
|  Chennai|          808|
|  Kolkata|          803|
|Hyderabad|          797|
|   Jaipur|          792|
|     Pune|          791|
|  Lucknow|          779|
|Bengaluru|          778|
+---------+-------------+



## Q7: How does the immutability of Spark DataFrames affect data cleaning operations?

Spark DataFrames are immutable. This means that an existing DataFrame cannot be modified directly.

Operations such as:

- Removing columns
- Renaming columns
- Filtering records
- Removing duplicates
- Changing data types

create a new DataFrame instead of modifying the original DataFrame.

For example:

df_unique = df.dropDuplicates()

## Q8: Filter a dataset where age is between 18 and 30 and subscription is Premium

Multiple filtering conditions can be combined using logical operators.

The between() function is used to select records where age is between 18 and 30, while another condition selects only Premium subscribers.

In [13]:

premium_users = df.filter(
    (col("age").between(18, 30)) &
    (col("subscription") == "Premium")
)

print(
    "Premium Users Between Age 18 and 30:",
    premium_users.count()
)

premium_users.select(
    "user_id",
    "age",
    "subscription",
    "region",
    "product_category"
).show(10, truncate=False)

Premium Users Between Age 18 and 30: 700
+-------+---+------------+-------+----------------+
|user_id|age|subscription|region |product_category|
+-------+---+------------+-------+----------------+
|USR1169|27 |Premium     |South  |Books           |
|USR1974|22 |Premium     |North  |Clothing        |
|USR0739|18 |Premium     |North  |Clothing        |
|USR2430|20 |Premium     |South  |Books           |
|USR1664|19 |Premium     |North  |Books           |
|USR1512|23 |Premium     |South  |Books           |
|USR0606|30 |Premium     |East   |Electronics     |
|USR0952|29 |Premium     |Central|Electronics     |
|USR2225|19 |Premium     |Central|Electronics     |
|USR0199|19 |Premium     |West   |Electronics     |
+-------+---+------------+-------+----------------+
only showing top 10 rows


## Q9: Why should null values be handled before performing mathematical aggregations such as sum() and avg()?

Null values can affect data quality and may result in incomplete or misleading calculations.

Before performing mathematical aggregations, missing values should be checked and handled appropriately.

Depending on the dataset and business requirement, null values can be:

- Removed using dropna()
- Replaced using fillna()
- Replaced with a suitable statistical value

In [14]:
df_price_filled = df.na.fill({
    "price": 0
})

print("Null Price Values After Filling:")

df_price_filled.select(
    count(
        when(col("price").isNull(), "price")
    ).alias("NullPriceCount")
).show()


Null Price Values After Filling:
+--------------+
|NullPriceCount|
+--------------+
|             0|
+--------------+



## Q10: Cast raw_timestamp to TimestampType and rename it to event_time

Schema modification is an important data transformation operation.

The cast() function is used to convert the raw_timestamp column into TimestampType.

The withColumnRenamed() function is then used to rename the column to event_time.


In [15]:
df_timestamp = df \
    .withColumn(
        "raw_timestamp",
        col("raw_timestamp").cast(TimestampType())
    ) \
    .withColumnRenamed(
        "raw_timestamp",
        "event_time"
    )

df_timestamp.select(
    "user_id",
    "transaction_date",
    "event_time"
).show(10, truncate=False)


+-------+----------------+-------------------+
|user_id|transaction_date|event_time         |
+-------+----------------+-------------------+
|USR0915|2025-04-25      |2025-04-25 13:02:00|
|USR0815|2025-08-27      |2025-08-27 00:48:00|
|USR0380|2025-10-12      |2025-10-12 01:46:00|
|USR1482|2026-01-23      |2026-01-23 07:49:00|
|USR0667|2025-04-14      |2025-04-14 20:04:00|
|USR2282|2025-09-08      |2025-09-08 12:17:00|
|USR1880|2025-11-19      |2025-11-19 17:16:00|
|USR2022|2026-02-13      |2026-02-13 21:27:00|
|USR1030|2026-01-26      |2026-01-26 08:49:00|
|USR2051|2026-03-22      |2026-03-22 19:12:00|
+-------+----------------+-------------------+
only showing top 10 rows


In [16]:
print("Schema After Timestamp Conversion and Column Renaming:")

df_timestamp.printSchema()


Schema After Timestamp Conversion and Column Renaming:
root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)



## Q11: Explain the Shuffle process that occurs during a grouping operation. Why is it considered a wide transformation?

A shuffle occurs when Spark redistributes data between different partitions.

Operations such as:

- groupBy()
- orderBy()
- join()


In [17]:
region_summary = df \
    .groupBy("region") \
    .agg(
        count("*").alias("total_records"),
        sum("sale_amount").alias("total_sales"),
        avg("sale_amount").alias("average_sales")
    ) \
    .orderBy(
        col("total_sales").desc()
    )

region_summary.show()

+-------+-------------+--------------------+------------------+
| region|total_records|         total_sales|     average_sales|
+-------+-------------+--------------------+------------------+
|  North|         1661|2.0860780199999973E7|12559.169295605041|
|  South|         1595|2.0488447860000025E7|12845.421855799388|
|   West|         1575|2.0160856620000012E7|12800.543885714293|
|Central|         1565|1.9553411510000024E7| 12494.19265814698|
|   East|         1604| 1.932196661999999E7|12046.113852867824|
+-------+-------------+--------------------+------------------+



## Q12: Remove rows where email is null OR username is an empty string

Data containing missing or empty values can affect data quality.

The following filtering operation keeps only records where:

- Email is not null
- Username is not an empty string


In [18]:
valid_user_data = df.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

print("Records Before Cleaning:", df.count())
print("Records After Cleaning:", valid_user_data.count())

print(
    "Invalid Records Removed:",
    df.count() - valid_user_data.count()
)


Records Before Cleaning: 8000
Records After Cleaning: 7850
Invalid Records Removed: 150


In [19]:
valid_user_data.select(
    "user_id",
    "email",
    "username"
).show(10, truncate=False)

+-------+--------------------+---------+
|user_id|email               |username |
+-------+--------------------+---------+
|USR0915|user245@example.com |user_768 |
|USR0815|user1308@example.com|user_3463|
|USR0380|user3764@example.com|user_4393|
|USR1482|user2371@example.com|user_654 |
|USR0667|user4991@example.com|user_1402|
|USR2282|user543@example.com |user_1729|
|USR1880|user4789@example.com|user_3510|
|USR2022|user4886@example.com|user_521 |
|USR1030|user2787@example.com|user_914 |
|USR2051|user1253@example.com|user_3064|
+-------+--------------------+---------+
only showing top 10 rows


## Q13: Calculate multiple statistics using the .agg() function

The agg() function allows multiple aggregation operations to be performed together.

The following statistics are calculated for the price column:

- Total number of price records
- Total price
- Average price
- Minimum price
- Maximum price


In [20]:
price_statistics = df_price_filled.agg(
    count("price").alias("total_records"),
    sum("price").alias("total_price"),
    avg("price").alias("average_price"),
    min("price").alias("minimum_price"),
    max("price").alias("maximum_price")
)

price_statistics.show()

+-------------+-------------------+-----------------+-------------+-------------+
|total_records|        total_price|    average_price|minimum_price|maximum_price|
+-------------+-------------------+-----------------+-------------+-------------+
|         8000|1.995120231000004E7|2493.900288750005|          0.0|      4999.88|
+-------------+-------------------+-----------------+-------------+-------------+



## Q14: What is the risk of using inferSchema=true with messy or inconsistent date formats?

The inferSchema=true option automatically determines the data type of each column by examining the dataset.

If a column contains inconsistent or messy date formats, Spark may:

- Infer the column as a string instead of a date
- Interpret values incorrectly
- Produce null values during later casting operations
- Cause data processing errors


In [21]:
final_pipeline = df \
    .dropDuplicates() \
    .na.fill({"price": 0}) \
    .groupBy("store_id") \
    .agg(
        sum("price").alias("total_revenue")
    ) \
    .orderBy(
        col("total_revenue").desc()
    )

final_pipeline.show()

+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
| STORE01|1113413.9000000004|
| STORE09|1084862.3099999987|
| STORE20|1062788.8499999999|
| STORE16|1060807.4799999995|
| STORE13|1057924.4300000002|
| STORE17|1044439.5900000001|
| STORE12|1025182.6499999998|
| STORE14|1013042.1599999996|
| STORE04| 997120.3200000001|
| STORE18| 994628.7199999999|
| STORE08| 987326.0100000004|
| STORE06| 986428.5300000004|
| STORE19| 983166.4999999999|
| STORE05|         976688.47|
| STORE10|  939563.240000001|
| STORE11|  935856.839999999|
| STORE02| 931231.8599999994|
| STORE07|         929647.53|
| STORE15| 917826.1799999994|
| STORE03| 909256.7399999993|
+--------+------------------+



In [22]:
store_revenue = df \
    .dropDuplicates() \
    .na.fill({"price": 0}) \
    .withColumn(
        "revenue",
        col("price") * col("quantity")
    ) \
    .groupBy("store_id") \
    .agg(
        sum("revenue").alias("total_revenue")
    ) \
    .orderBy(
        col("total_revenue").desc()
    )

store_revenue.show()


+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
| STORE01| 6218414.460000001|
| STORE13| 5947923.250000003|
| STORE09|5873963.3599999985|
| STORE17|        5863260.78|
| STORE04| 5727803.660000002|
| STORE20| 5720021.469999995|
| STORE06|        5688714.75|
| STORE16| 5646370.039999995|
| STORE14| 5610549.749999996|
| STORE07| 5481854.479999998|
| STORE19|        5474022.99|
| STORE12| 5442089.569999996|
| STORE08| 5359301.339999995|
| STORE05| 5344084.609999998|
| STORE11| 5309063.780000001|
| STORE18| 5258705.299999997|
| STORE10|5203200.8199999975|
| STORE03| 5050459.909999997|
| STORE02| 5039895.309999998|
| STORE15| 4965616.619999999|
+--------+------------------+



## Complete Data Processing Pipeline

The complete Spark data processing workflow performed in this assignment includes:

1. Creating a SparkSession.
2. Loading a CSV dataset.
3. Exploring the dataset.
4. Displaying records, column names, and schema.
5. Understanding the limitations of MapReduce.
6. Understanding Spark in-memory computing.
7. Removing duplicate records.
8. Filtering data according to conditions.
9. Checking and handling null values.
10. Applying conditions to aggregated results.
11. Understanding DataFrame immutability.
12. Casting column data types.
13. Renaming columns.
14. Removing inconsistent data.
15. Performing multiple aggregation operations.
16. Using groupBy() for data analysis.
17. Understanding wide transformations and shuffle operations.
18. Building a complete data processing pipeline.
19. Saving the final results.


In [23]:
 %pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import os

# Create Output folder if it does not exist
os.makedirs("../Output", exist_ok=True)

# Convert Spark DataFrame to Pandas
final_results = store_revenue.toPandas()

# Save CSV inside Output folder
final_results.to_csv("../Output/results.csv", index=False)

print("Results Saved Successfully")

Results Saved Successfully
